# Round 1 — 재매핑 데이터 기반 초기 학습
- 베이스 모델: `yolov8s.pt`
- 데이터: `data_remapped` (no_helmet / ok)
- 저장 위치: `runs2/round1`
- MLflow: 자동 로깅

In [ ]:
import mlflow
from ultralytics import YOLO

# ── 경로 설정 ──────────────────────────────────────
DATA_YAML    = '/mnt/ml/helmet_project/data/data_remapped/data.yaml'
PROJECT_DIR  = '/mnt/ml/helmet_project/experiments/runs2'
RUN_NAME     = 'round1'
BASE_MODEL   = '/mnt/ml/helmet_project/models/pretrained/yolov8s.pt'

# ── MLflow 설정 ────────────────────────────────────
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('helmet_detection')

with mlflow.start_run(run_name=RUN_NAME):

    # 파라미터 수동 기록
    mlflow.log_params({
        'base_model' : BASE_MODEL,
        'data'       : DATA_YAML,
        'epochs'     : 50,
        'imgsz'      : 640,
        'batch'      : 16,
        'round'      : 1,
        'cls_weight' : 1.0,
    })

    model = YOLO(BASE_MODEL)

    results = model.train(
        data     = DATA_YAML,
        epochs   = 50,
        imgsz    = 640,
        batch    = 16,
        device   = 0,
        project  = PROJECT_DIR,
        name     = RUN_NAME,
        exist_ok = True,
    )

    # 최종 메트릭 기록
    metrics = results.results_dict
    mlflow.log_metrics({
        'mAP50'     : metrics.get('metrics/mAP50(B)', 0),
        'mAP50_95'  : metrics.get('metrics/mAP50-95(B)', 0),
        'precision' : metrics.get('metrics/precision(B)', 0),
        'recall'    : metrics.get('metrics/recall(B)', 0),
    })

    # best.pt 아티팩트 저장
    best_pt = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
    mlflow.log_artifact(best_pt, artifact_path='weights')

    print(f'\n✅ Round1 완료')
    print(f'   best.pt: {best_pt}')
    print(f'   mAP50  : {metrics.get("metrics/mAP50(B)", 0):.4f}')

In [ ]:
# 검증 (선택)
from ultralytics import YOLO

model = YOLO('/mnt/ml/helmet_project/experiments/runs2/round1/weights/best.pt')
val   = model.val(data='/mnt/ml/helmet_project/data/data_remapped/data.yaml', split='test')
print(val)